# Diseño de Muro de Concreto - Flexocompresión (NTC 2023)
Este programa calcula la profundidad del eje neutro ($c$) y el área de acero ($A_s$) requerida para la sección transversal especificada.

**Consideraciones:**
- $f'c = 250 \text{ kg/cm}^2$
- $fy = 4200 \text{ kg/cm}^2$
- Norma: NTC-Concreto 2023 (CDMX)
- Altura del muro = 4 m

In [ ]:
import numpy as np
from scipy.optimize import fsolve

# 1. DATOS DE ENTRADA
fc = 350      # kg/cm2
fy = 4200     # kg/cm2
Es = 2e6      # kg/cm2

# Cargas Últimas
Pu = 1694 * 1000          # kg
Mu = 3933.96 * 1000 * 100 # kg-cm
Vu = 1187.75 * 1000       # kg

# Geometría Base
b_alma = 50   # cm
h_alma = 428  # cm
b_patin = 90  # cm
h_patin = 150 # cm
L_total = h_alma + h_patin # 578 cm

# Elemento extremo (Caso 2)
bc = 50       # cm
hc = 150      # cm

f_bic = 0.85 * fc
beta1 = 0.85

print("Datos inicializados correctamente.")

Datos inicializados correctamente.


## Caso 1: Compresión en el elemento de 90x150 cm (El Patín)

In [2]:
def equilibrio_caso1(vars):
    c, As = vars
    a = beta1 * c
    
    if a <= h_patin:
        Cc = f_bic * b_patin * a
        yc = a / 2
    else:
        Cc1 = f_bic * b_patin * h_patin
        Cc2 = f_bic * b_alma * (a - h_patin)
        Cc = Cc1 + Cc2
        yc = (Cc1 * (h_patin/2) + Cc2 * (h_patin + (a-h_patin)/2)) / Cc
        
    d = L_total - 10
    es = 0.003 * (d - c) / c
    fs = min(Es * es, fy)
    
    eq1 = Pu - (Cc - As * fs)
    cg = L_total / 2
    eq2 = Mu - (Cc * (cg - yc) + As * fs * (d - cg))
    
    return [eq1, eq2]

res_caso1 = fsolve(equilibrio_caso1, [150, 200])
c_1, As_1 = res_caso1
print(f"--- CASO 1: COMPRESIÓN EN EL PATÍN (90x150) ---")
print(f"Profundidad del Eje Neutro (c): {c_1:.2f} cm")
print(f"Área de acero a tensión requerida (As): {As_1:.2f} cm²")

--- CASO 1: COMPRESIÓN EN EL PATÍN (90x150) ---
Profundidad del Eje Neutro (c): 101.50 cm
Área de acero a tensión requerida (As): -10.48 cm²


## Caso 2: Compresión en el extremo del alma con elemento extremo (bc=50 cm, hc=150 cm)
Se considera explícitamente el bloque de compresión segmentado tomando en cuenta el elemento de borde o extremo de 50x150 cm.

In [3]:
def equilibrio_caso2(vars):
    c, As = vars
    a = beta1 * c
    
    # Fuerza de compresión en el concreto (Cc) segmentada con el elemento extremo
    if a <= hc:
        # El bloque equivalente está completamente dentro del elemento extremo
        Cc = f_bic * bc * a
        yc = a / 2
    elif a <= h_alma:
        # El bloque equivalente abarca el elemento extremo y parte del resto del alma
        Cc1 = f_bic * bc * hc
        Cc2 = f_bic * b_alma * (a - hc)
        Cc = Cc1 + Cc2
        yc = (Cc1 * (hc/2) + Cc2 * (hc + (a-hc)/2)) / Cc
    else:
        # El bloque de compresión llega hasta el patín opuesto
        Cc1 = f_bic * bc * hc
        Cc2 = f_bic * b_alma * (h_alma - hc)
        Cc3 = f_bic * b_patin * (a - h_alma)
        Cc = Cc1 + Cc2 + Cc3
        yc = (Cc1 * (hc/2) + Cc2 * (hc + (h_alma-hc)/2) + Cc3 * (h_alma + (a-h_alma)/2)) / Cc
        
    d = L_total - 10 # recubrimiento supuesto
    
    es = 0.003 * (d - c) / c
    fs = min(Es * es, fy)
    
    # Ecuaciones de equilibrio
    eq1 = Pu - (Cc - As * fs)
    
    cg = L_total / 2
    eq2 = Mu - (Cc * (cg - yc) + As * fs * (d - cg))
    
    return [eq1, eq2]

res_caso2 = fsolve(equilibrio_caso2, [150, 200])
c_2, As_2 = res_caso2
print(f"--- CASO 2: COMPRESIÓN EN EL EXTREMO DEL ALMA (Elemento 50x100 cm) ---")
print(f"Profundidad del Eje Neutro (c): {c_2:.2f} cm")
print(f"Área de acero a tensión requerida (As): {As_2:.2f} cm²")

--- CASO 2: COMPRESIÓN EN EL EXTREMO DEL ALMA (Elemento 50x100 cm) ---
Profundidad del Eje Neutro (c): 198.22 cm
Área de acero a tensión requerida (As): 22.91 cm²
